# Zero-Shot SSDU (ZS-SSL) 데모 — 단일 스캔 자기지도 복원

학습셋 없이 **테스트 스캔 1장만으로** 학습. 획득 마스크 Ω를:
- **Γ (validation)**: Ω에서 떼어낸 holdout → early stopping
- 나머지 **Ω\Γ**: 매 step **Θ(DC)/Λ(loss)** 로 재분할 → Λ k-space loss로 학습
- 최종 추론은 **전체 Ω** 데이터 일관성
- **비교 기준(GT) = 우리 SENSE combination** (recon이 SENSE 도메인 출력).

In [ ]:
import os, sys, time, glob
sys.path.insert(0, '/home/sonwonjun/research/MRRecon/code')
import numpy as np, torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from mrrecon.config import Config
from mrrecon.data.loaders import read_slice
from mrrecon.data.masks import undersampling_mask
from mrrecon.data.datasets import ZeroShotDataset
from mrrecon.models import build_unrolled
from mrrecon.losses import MixL1L2Loss
from mrrecon.metrics import all_metrics, match_scale
from mrrecon.data.transforms import sense_combine_np
from mrrecon.core.common import center_crop
from mrrecon.core.inference import recon_unrolled

cfg=Config(); cfg.tissue='knee'; cfg.device='cuda' if torch.cuda.is_available() else 'cpu'
cfg.model='ssdu'; cfg.nb_unroll_blocks=5; cfg.res_blocks=4; cfg.cg_iter=5; cfg.mu=0.05
cfg.acc_rate=4; cfg.acs_lines=24; cfg.mask_type='vds_lustig'; cfg.rho=0.4
cfg.zs_val_rho=0.2; cfg.zs_num_splits=8; cfg.lr=5e-4; cfg.epochs=40; cfg.zs_patience=15; cfg.crop_size=320; cfg.seed=1234
dev=torch.device(cfg.device); print('device',dev)

## 0. 데이터 먼저 보기 — k-space / mask / undersampled (+ zero-filled SSIM·PSNR)

In [ ]:
# 한 test 스캔 + 언더샘플 마스크 Omega
files=sorted(glob.glob('/mnt/d/research/MRRecon/knee/test/*.h5'))
sf=files[len(files)//2]
kspace,sens,rss=read_slice(sf,crop_size=cfg.crop_size)
H,W=kspace.shape[1:]
omega=undersampling_mask((H,W),cfg.acc_rate,cfg.acs_lines,cfg.mask_type,rng=np.random.default_rng(cfg.seed),vds_power=cfg.vds_power)

sense_gt=np.abs(sense_combine_np(kspace,sens))             # GT = SENSE combination (fully-sampled)
zf=np.abs(sense_combine_np(kspace*omega[None],sens))       # zero-filled (undersampled) SENSE
gtc,zfc=center_crop(sense_gt),center_crop(zf)
mzf=all_metrics(gtc,match_scale(gtc,zfc))                  # zero-filled vs SENSE GT
line=lambda m:m[m.shape[0]//2]

fig,ax=plt.subplots(1,4,figsize=(19,5))
ax[0].imshow(np.log(np.abs(kspace[7])+1e-9),cmap='gray'); ax[0].set_title('k-space (coil7) |k|log'); ax[0].axis('off')
ax[1].imshow(omega,cmap='gray',aspect='auto'); ax[1].set_title(f'mask Omega (acc {cfg.acc_rate}x, {int(line(omega).sum())}/{W})'); ax[1].axis('off')
ax[2].imshow(zfc,cmap='gray',vmax=0.6*gtc.max()); ax[2].set_title(f'undersampled (zero-filled)\nSSIM={mzf["ssim"]:.3f}  PSNR={mzf["psnr"]:.2f}'); ax[2].axis('off')
ax[3].imshow(gtc,cmap='gray',vmax=0.6*gtc.max()); ax[3].set_title('SENSE combination = GT'); ax[3].axis('off')
plt.suptitle(f'input data: {sf.split(chr(47))[-1]}'); plt.tight_layout(); plt.show()
print(f'zero-filled vs SENSE-GT : SSIM={mzf["ssim"]:.4f}  PSNR={mzf["psnr"]:.3f}  NMSE={mzf["nmse"]:.5f}')


## 1. 마스크 분할: Ω → Γ(val) + (Θ,Λ)

In [ ]:
ds=ZeroShotDataset(cfg,kspace,sens,omega)
b=ds[0]; theta=b['trn_mask'][0].numpy(); lam=b['loss_mask'][0].numpy()
gamma=ds.val_mask; trainp=ds.train_mask
fig,ax=plt.subplots(2,1,figsize=(13,6))
for nm,m in [('Omega',omega),('Gamma(val)',gamma),('Omega\\Gamma(train)',trainp),('Theta(DC)',theta),('Lambda(loss)',lam)]:
    ax[0].plot(line(m),label=nm,alpha=0.7)
ax[0].legend(fontsize=8); ax[0].set_ylim(-0.1,1.1); ax[0].set_title('phase patterns (1D)'); ax[0].set_xlabel('phase col')
ax[1].imshow(np.stack([line(omega),line(gamma),line(theta),line(lam)]),aspect='auto',cmap='gray')
ax[1].set_yticks(range(4)); ax[1].set_yticklabels(['Omega','Gamma','Theta','Lambda']); ax[1].set_xlabel('phase col')
plt.tight_layout(); plt.show()
print(f'Omega={int(line(omega).sum())} Gamma={int(line(gamma).sum())} Theta={int(line(theta).sum())} Lambda={int(line(lam).sum())}')

## 2. 모델 + ZS-SSL 학습 (Γ early stopping)

In [ ]:
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
model=build_unrolled(cfg).to(dev); opt=torch.optim.Adam(model.parameters(),lr=cfg.lr); loss_fn=MixL1L2Loss().to(dev)
dl=DataLoader(ds,batch_size=1,shuffle=False,num_workers=0)
def eval_sense(model):
    ref,zfb,rec=recon_unrolled(model,kspace,sens,omega,dev)   # ref=SENSE GT, zfb=zero-filled
    rc,rf,zc=center_crop(np.abs(rec)),center_crop(ref),center_crop(zfb)
    return (all_metrics(rf,match_scale(rf,rc)), all_metrics(rf,match_scale(rf,zc)), np.abs(rec), ref, zfb)

def to_dev(b): return (b['x_in'].to(dev),b['sens_maps'].to(dev),b['ref_kspace'].to(dev),b['trn_mask'].to(dev),b['loss_mask'].to(dev))
@torch.no_grad()
def val_kloss():
    model.eval(); vb={k:(v.unsqueeze(0) if torch.is_tensor(v) else v) for k,v in ds.val_sample().items()}
    x,s,rk,tr,lm=to_dev(vb); _,_,nw=model(x,s,tr,lm); return loss_fn(nw,rk).item()
hist=[]; best=1e9; best_state=None; since=0; t0=time.time()
for ep in range(cfg.epochs):
    model.train(); el=0;n=0
    for bb in dl:
        x,s,rk,tr,lm=to_dev(bb); _,_,nw=model(x,s,tr,lm); loss=loss_fn(nw,rk)
        opt.zero_grad(); loss.backward(); opt.step(); el+=loss.item(); n+=1
    el/=n; vl=val_kloss(); mr,_,_,_,_=eval_sense(model)
    hist.append({'ep':ep+1,'train':el,'val':vl,'ssim':mr['ssim'],'psnr':mr['psnr']})
    if vl<best: best=vl; best_state={k:v.detach().clone() for k,v in model.state_dict().items()}; since=0
    else: since+=1
    if (ep+1)%5==0 or ep==0: print(f'ep{ep+1:3d} train={el:.4f} val={vl:.4f}  recon SSIM={mr["ssim"]:.4f} PSNR={mr["psnr"]:.2f}')
    if since>=cfg.zs_patience: print(f'early stop @ ep{ep+1}'); break
print(f'done {time.time()-t0:.0f}s, best val={best:.4f}')
ep=[h['ep'] for h in hist]
fig,ax=plt.subplots(1,3,figsize=(16,4))
ax[0].plot(ep,[h['train'] for h in hist],label='train(Lambda)'); ax[0].plot(ep,[h['val'] for h in hist],label='val(Gamma)'); ax[0].legend(); ax[0].set_title('k-space loss'); ax[0].grid(alpha=0.3)
ax[1].plot(ep,[h['ssim'] for h in hist],'C2'); ax[1].set_title('recon SSIM vs SENSE-GT'); ax[1].grid(alpha=0.3)
ax[2].plot(ep,[h['psnr'] for h in hist],'C3'); ax[2].set_title('recon PSNR vs SENSE-GT'); ax[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. 최종 복원 (best model, 전체 Ω) vs SENSE-GT

In [ ]:
model.load_state_dict(best_state)
mr,mz,rec,ref,zfb=eval_sense(model)
gtc=center_crop(ref); zc=center_crop(np.abs(zfb)); rc=center_crop(np.abs(rec))
fig,ax=plt.subplots(1,3,figsize=(15,5))
ax[0].imshow(gtc,cmap='gray',vmax=0.6*gtc.max()); ax[0].set_title('SENSE GT'); ax[0].axis('off')
ax[1].imshow(match_scale(gtc,zc),cmap='gray',vmax=0.6*gtc.max()); ax[1].set_title(f'zero-filled\nSSIM={mz["ssim"]:.3f}  PSNR={mz["psnr"]:.2f}'); ax[1].axis('off')
ax[2].imshow(match_scale(gtc,rc),cmap='gray',vmax=0.6*gtc.max()); ax[2].set_title(f'ZS-SSL recon\nSSIM={mr["ssim"]:.3f}  PSNR={mr["psnr"]:.2f}'); ax[2].axis('off')
plt.suptitle('ZS-SSL vs SENSE-GT'); plt.tight_layout(); plt.show()
print(f'zero-filled : SSIM={mz["ssim"]:.4f}  PSNR={mz["psnr"]:.3f}')
print(f'ZS-SSL recon: SSIM={mr["ssim"]:.4f}  PSNR={mr["psnr"]:.3f}  NMSE={mr["nmse"]:.5f}')

**요약**: ZS-SSL = 단일 스캔, Ω→Γ + Θ/Λ, Γ k-space loss로 early stop, 전체 Ω로 최종 추론. GT는 SENSE combination, zero-filled/recon 모두 SENSE-GT 기준 SSIM·PSNR.